# 임베딩 실습 3일차: LLM 이름 초안 · 이름 비교 · 팀 라운드 요약

> 2일차에서 확정한 토픽에 대해 **LLM 이름 초안**을 만들고, 우리가 정한 이름과 비교한 뒤 **팀장 요약**을 정리합니다.

## 전체 흐름

```
2일차 결과 (토픽 · 상위 단어 · 대표 논문)
   ↓ ① LLM(Qwen3-4B) 이름 초안, 온도 0
   ↓ ② 우리 이름과 비교 (무작위 10건 ○/×)
   ↓ ③ 팀 라운드: 묶음 기준(생물 · 관리·정책 · 환경·기후)으로 요약
```

## 1. 준비

- 2일차 노트북을 먼저 실행해 `topics`, `words`, `W`, `lab`, `red`, `df`가 있어야 해요.
- 런타임이 새로 시작됐다면 2일차의 준비 셀(데이터·벡터·UMAP·HDBSCAN)을 먼저 실행하세요.

In [ ]:
!pip -q install transformers accelerate
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

name = 'Qwen/Qwen3-4B-Instruct-2507'
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, device_map='cuda')
print('모델 준비 완료')

## 2. LLM 이름 초안 만들기

**방법**
- 입력: 상위 단어 10개 + 중심에 가장 가까운 대표 논문 4편
- 설정: 온도 0(`do_sample=False`) → 같은 입력에 항상 같은 답
- 교안과 같은 형식(이름 · 설명)으로 받아요.

**주의**: LLM 초안은 **초안**이에요. 우리 이름과 같이 무작위 10건으로 확인해야 해요.

In [ ]:
import numpy as np

t = 3
k = topics.index(t)
top10 = words[np.argsort(-W[k])[:10]]

idx_t = np.where(lab == t)[0]
cen = red[idx_t].mean(axis=0)
order = idx_t[np.argsort(np.linalg.norm(red[idx_t] - cen, axis=1))][:4]
titles = [df['title'].iloc[i] for i in order]

system = '당신은 학술 논문의 주제를 분석하는 보조입니다.'
user = ('상위어: ' + ', '.join(top10) + '\n\n대표 논문 제목:\n'
        + '\n'.join('- ' + x for x in titles)
        + '\n\n형식:\n이름: (10자 안팎 명사구)\n설명: (한 문장)')
msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]

text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tok(text, return_tensors='pt').to('cuda')
out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
print(tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

### 2-1. LLM 초안 결과 (토픽 3)

| 초안 이름 | 판단 |
|---|---|
| 해양 생물 다양성 | 9/10 통과, 하지만 너무 넓어 다른 테마와 구별되지 않음 |
| 번식 성과 평가 | 0/10 실패. 대표 논문 4편의 번식 단어에 맞춰 생긴 이름 |
| 생태적 영향 분석 | 3/10 실패. 무엇의 영향인지 불분명 |

**최종 이름 (우리 판단)**: **상어·가오리 연구 (개체군·어획·생태)** — 9/10 통과, 다른 테마와 구별됨

## 3. 이름 비교 (무작위 10건 ○/×)

같은 10건(2일차 시드 0 표본)을 두 이름으로 나눠 판단했어요.

| # | 논문 (앞부분) | 해양 생물 다양성 | 번식 성과 평가 | 생태적 영향 분석 | 상어·가오리 연구 |
|---|---|:---:|:---:|:---:|:---:|
| 1 | 상어 새 종 (고양이상어) | ○ | × | × | ○ |
| 2 | 인도 가오리 목록 | ○ | × | × | ○ |
| 3 | 가오리 체중 관계 | ○ | × | × | ○ |
| 4 | 백상아리 불법 어획 | ○ | × | ○ | ○ |
| 5 | 상어 개체군 추세 | ○ | × | ○ | ○ |
| 6 | 남대서양 상어 어획 | ○ | × | ○ | ○ |
| 7 | 백상아리 수영과 에너지 | ○ | × | × | ○ |
| 8 | 열대송어 계절 이동 | ○ | × | × | × |
| 9 | 고래상어 나이 측정 | ○ | × | × | ○ |
| 10 | 상어 이빨 광물화 | ○ | × | × | ○ |
| **합계** | | **9/10** | **0/10** | **3/10** | **9/10** |

> 8번(열대송어)은 상어가 아니라서 우리 이름에서는 ×예요. 해양 생물 다양성에는 ○로 넣었어요.

## 4. 팀 라운드: 팀장 요약 (C안, 묶음 기준)

**질문**: 테마 12개를 그대로 보여 줄지, 묶음으로 보여 줄지
**선택**: C안 — 세 묶음으로 나눠 보여 줌

| 묶음 | 포함 테마 | 편수 | 비율 |
|---|---|---:|---:|
| 생물 연구 | 상어·가오리 4,928 · 연체류 3,442 | 8,370 | 28.45% |
| 관리·정책 | 보호구역 532 · 블루 이코노미 380 · 해양공간계획 283 · 중국 중심 해양 경제 457 · 발트해 생태와 관리 61 · 생태계 기반 관리와 위험 평가 237 · 해양 과학과 정책 318 | 2,268 | 7.71% |
| 환경·기후 | 해양 환경 연구와 영향 480 · 내해·하구 수질 110 · 연안 기후 적응 254 | 844 | 2.87% |
| 토픽 24 중 하위에 안 속함 | 연안·해양 관리 토픽의 나머지 | 612 | 2.08% |
| 이상치 (주제 없음) | — | 9,239 | 31.41% |
| 미분류 | 언어 묶음, 데이터 문구 토픽, 제외한 토픽(38번 등) | 8,084 | 27.48% |
| **합계** | | **29,417** | **100%** |

**주의**
- 관리·정책의 발트해·생태계 기반·과학 정책 3개는 토픽 24 안만 따로 나눈 결과라서 기준이 조금 달라요.
- 모든 숫자는 우리 표본 기준이에요. 연도 추세나 전 세계 순위로 말하면 안 돼요.

**팀장께 보낼 문장**
> 분석 표본 29,417편에서 생물 연구 묶음이 약 28%로 가장 크고, 관리·정책 묶음이 약 8%, 환경·기후 묶음이 약 3%였습니다. 나머지는 주제가 없거나 분류에서 제외되었습니다.

## 5. 정리와 보내기 전 체크

- [x] LLM 초안을 만들고, 우리 이름과 무작위 10건으로 비교했다
- [x] 우리 이름(상어·가오리 연구)이 9/10으로 통과하고, 다른 테마와 구별된다
- [x] 팀 요약을 C안(세 묶음)으로 정리했다
- [ ] 비율 분모가 전체 29,417편인지 확인
- [ ] "가장 많다"를 표본 결과로 한정했는지 확인
- [ ] 재현 설정(임베딩, UMAP 이웃 15·5차원, HDBSCAN 최소 군집 크기 50, 시드 0)을 기록